In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [2]:
import sys
sys.path.append("/Users/mariahloehr/IICD/IICD/feature_importance")

In [3]:
import locomp
from locomp import *
from locomp.MLmodels import *
from locomp.util_locomp import *
import itertools
import importlib
from sklearn.base import BaseEstimator, RegressorMixin, clone
import itertools
from functools import partial
import multiprocessing as mp
import re

In [4]:
# Load data
df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Data/cell_cycle_tidied.csv")

#df['phase'] = df['phase'].replace({'M': 'G2'})

# Separate features and target
X = df.drop(columns=['phase', 'age', 'PHATE_1', 'PHATE_2'])  # exclude phase and age
y = df['age']  # target is now age

feature_names = X.columns.tolist()
X = X.to_numpy()
y = y.to_numpy()

# Split data into train and test sets (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=949)

In [5]:
def DecisionTreeReg(X,Y,X1):
    tree = DecisionTreeRegressor(max_depth = 20, 
                                 max_features=150,
                                 random_state=949
                                 ).fit(X,Y)
    return tree.predict(X1)

In [6]:
J1 = 0
J2 = 1
m_ratio = 0.5
n_ratio = 0.5
B = 5000
fit_func = DecisionTreeReg

In [56]:
predictions, in_mp_obs, in_mp_feature = predictMPReg(X_train, y_train, X_test, n_ratio, m_ratio, B, fit_func)

# Aggregate predictions from all minipatch models
mean_pred = np.mean(predictions, axis=0)

In [57]:
rmse = root_mean_squared_error(y_test, mean_pred)
print("Minipatch Ensemble RMSE:", rmse)

Minipatch Ensemble RMSE: 1.4627887437277034


In [8]:
# Your parameter ranges
n_ratios = np.arange(0.1, 1.1, 0.1)
m_ratios = np.arange(0.1, 1.1, 0.1)

# Initialize matrix to store RMSE values
rmse_matrix = np.zeros((len(n_ratios), len(m_ratios)))

# Loop through all combinations of n_ratio and m_ratio
for i, n_ratio in enumerate(n_ratios):
    for j, m_ratio in enumerate(m_ratios):
        predictions, in_mp_obs, in_mp_feature = predictMPReg(X_train, y_train, X_test, n_ratio, m_ratio, B, fit_func)
        mean_pred = np.mean(predictions, axis=0)
        rmse = root_mean_squared_error(y_test, mean_pred)
        rmse_matrix[i, j] = rmse

# Plot the heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(rmse_matrix, annot=True, fmt=".3f", xticklabels=m_ratios, yticklabels=n_ratios, cmap="viridis")
plt.xlabel("m_ratio (feature sampling)")
plt.ylabel("n_ratio (observation sampling)")
plt.title("RMSE Heatmap of MiniPatchReg for n_ratio vs m_ratio")
plt.show()

KeyboardInterrupt: 

In [58]:
# For test set
y = df['age']  # (reversing making it a numpy array)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=949)

df_test = pd.DataFrame({
    'true_age': y_test,
    'pred_age': mean_pred,
    'phase': df.loc[y_test.index, 'phase']  # get phase for train samples
})

rmse_per_phase_test = df_test.groupby('phase').apply(
    lambda x: root_mean_squared_error(x['true_age'], x['pred_age'])
)

print("\nRMSE per phase (Test):")
print(rmse_per_phase_test)


RMSE per phase (Test):
phase
G0    1.623708
G1    1.209423
G2    1.534784
M     5.632493
S     1.410828
dtype: float64


/var/folders/1s/bvxr71hj0hqgyk_jk6k7wkm80000gn/T/ipykernel_30097/2290763584.py:11: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  rmse_per_phase_test = df_test.groupby('phase').apply(


In [52]:
# === 0.2 Load existing results DataFrame ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/minipatch_results.csv", index_col=0)

# === Set values ===
model_name = "Decision Tree MP"  # or whatever is appropriate
results_df.loc[model_name, 'Overall'] = rmse

# Fill in per-phase RMSEs
for phase in ['G0', 'G1', 'G2', 'M', 'S']:
    if phase in rmse_per_phase_test.index:
        results_df.loc[model_name, phase] = rmse_per_phase_test[phase]

# === Save updated results ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/minipatch_results.csv")

In [ ]:
# === 0.5 Load existing results DataFrame ===
results_df = pd.read_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/.5_minipatch_results.csv", index_col=0)

# === Set values ===
model_name = "Decision Tree MP"  # or whatever is appropriate
results_df.loc[model_name, 'Overall'] = rmse

# Fill in per-phase RMSEs
for phase in ['G0', 'G1', 'G2', 'M', 'S']:
    if phase in rmse_per_phase_test.index:
        results_df.loc[model_name, phase] = rmse_per_phase_test[phase]

# === Save updated results ===
results_df.to_csv("/Users/mariahloehr/IICD/IICD/Bar Plot/.5_minipatch_results.csv")